# Four-Stage Hierarchical CNN Experiment

This notebook follows the same style as your hierarchy / multibranch experiments, but the target logic is now:

1. **Target vs Non-Target** on all sequences
2. **Orientation** on Target sequences
3. **Gesture Action** on Target sequences
4. **Gesture Position** on Target sequences

The final Target gesture is reconstructed from:

```text
gesture_position + " - " + gesture_action
```

The final competition-style hierarchy prediction is:

```text
if Target: reconstructed gesture
else: Non-Target
```

In [ ]:
from __future__ import annotations

import os
import json
import joblib
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
except Exception as exc:
    BayesSearchCV = None
    Categorical = None
    Integer = None
    Real = None
    print("BayesSearchCV unavailable. Use search_mode='grid'.", exc)

import data_utils
import utils_hierarchy_position_aug as utils

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
# ============================================================
# Config
# ============================================================

search_mode = "grid"      # "grid" or "bayesian"
random_state = 42
# ============================================================
# Data size control
# ============================================================

holdout_size = 0.2          # final untouched test set
use_train_subset = True     # True = train on a smaller subset
train_sequence_frac = 0.2   # use 20% of the training split
n_cv_splits = 3
n_iter_bayes = 12
n_jobs = 1

pipe_name = "sequence_builder"
classifier_name = "classifier"
corrector_name = "orientation_corrector"

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Reuse / loading options
# ============================================================

# Per model options:
# "search"      = run Bayes/Grid search as normal
# "best_params" = load params from a previous summary CSV, then refit on current train split
# "model"       = load a previously fitted pipeline from joblib, no refit
model_run_mode = {
    "target": "search",
    "orientation": "search",
    "action": "search",
    "position": "search",
}

# Used when model_run_mode[...] == "best_params"
# Point this at a previous summary_four_stage_hierarchy_cnn_*.csv
previous_summary_path = None

# Used when model_run_mode[...] == "model"
previous_model_paths = {
    "target": None,
    "orientation": None,
    "action": None,
    "position": None,
}

# Saves fitted best pipelines after search / best_params refit.
# These can later be used with model_run_mode[...] = "model".
save_fitted_models = True

# If True, CV result CSVs are still written for loaded/refit models.
save_loaded_model_stub_results = True

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
print("timestamp:", timestamp)
print("model_run_mode:", model_run_mode)

In [ ]:
# ============================================================
# Load data
# ============================================================

data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

print("raw_train_df:", raw_train_df.shape)
print("train_demo_df:", train_demo_df.shape)
print("train sequences:", raw_train_df["sequence_id"].nunique())
print("subjects:", raw_train_df["subject"].nunique())

In [ ]:
# ============================================================
# Base dataframe + helper targets
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df.loc[:, "gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df.loc[:, "gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df.loc[:, "is_target"] = train_df["sequence_type"].eq("Target").astype(int)

train_df.loc[:, "hierarchical_gesture"] = np.where(
    train_df["sequence_type"].eq("Target"),
    train_df["gesture"],
    "Non-Target",
)

print("sequence_type counts")
print(train_df.drop_duplicates("sequence_id")["sequence_type"].value_counts())

print("\ngesture_action classes")
print(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_action"].value_counts())

print("\ngesture_position classes")
print(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_position"].value_counts())

print("\norientation classes")
print(train_df.loc[train_df["sequence_type"].eq("Target"), "orientation"].value_counts())

In [ ]:
# ============================================================
# Mapping check: gesture_position + gesture_action -> original gesture
# ============================================================

target_map_df = (
    train_df
    .loc[train_df["sequence_type"].eq("Target")]
    .drop_duplicates(["gesture_position", "gesture_action", "gesture"])
    [["gesture_position", "gesture_action", "gesture"]]
    .copy()
)

mapping_check = (
    target_map_df
    .groupby(["gesture_position", "gesture_action"])["gesture"]
    .nunique()
    .reset_index(name="n_gesture_labels")
)

display(mapping_check.sort_values("n_gesture_labels", ascending=False))

if not mapping_check["n_gesture_labels"].eq(1).all():
    print("WARNING: gesture_position + gesture_action does not uniquely map to gesture.")

position_action_lookup = {
    (row.gesture_position, row.gesture_action): row.gesture
    for row in target_map_df.itertuples(index=False)
}

most_common_target_gesture = (
    train_df
    .loc[train_df["sequence_type"].eq("Target")]
    .drop_duplicates("sequence_id")["gesture"]
    .mode()
    .iloc[0]
)

print("lookup size:", len(position_action_lookup))
print("fallback target gesture:", most_common_target_gesture)

In [ ]:
# ============================================================
# Subject holdout split + optional train subset
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

# one row per sequence for splitting
seq_meta = (
    train_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "is_target"]]
    .reset_index(drop=True)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=holdout_size,
    random_state=random_state,
)

train_seq_idx, holdout_seq_idx = next(
    splitter.split(
        seq_meta,
        y=seq_meta["is_target"],
        groups=seq_meta["subject"],
    )
)

train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]

train_model_df = train_df.loc[
    train_df["sequence_id"].isin(train_seq_ids)
].copy()

holdout_df = train_df.loc[
    train_df["sequence_id"].isin(holdout_seq_ids)
].copy()


# ============================================================
# Optional train subset AFTER holdout
# ============================================================

if use_train_subset:
    train_seq_meta = (
        train_model_df
        .drop_duplicates("sequence_id")
        [["sequence_id", "subject", "sequence_type", "gesture", "is_target"]]
        .reset_index(drop=True)
    )

    sampled_train_seq_ids = (
        train_seq_meta
        .groupby("sequence_type", group_keys=False)
        .sample(
            frac=train_sequence_frac,
            random_state=random_state,
        )["sequence_id"]
    )

    train_model_df = train_model_df.loc[
        train_model_df["sequence_id"].isin(sampled_train_seq_ids)
    ].copy()


# ============================================================
# Target-only views for specialist models
# ============================================================

target_only_train_df = train_model_df.loc[
    train_model_df["sequence_type"].eq("Target")
].copy()

target_only_holdout_df = holdout_df.loc[
    holdout_df["sequence_type"].eq("Target")
].copy()


# ============================================================
# Split checks
# ============================================================

print("full sequences:", train_df["sequence_id"].nunique())
print("train sequences:", train_model_df["sequence_id"].nunique())
print("holdout sequences:", holdout_df["sequence_id"].nunique())

print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())
print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())

print("train subjects:", train_model_df["subject"].nunique())
print("holdout subjects:", holdout_df["subject"].nunique())

print(
    "subject overlap:",
    len(set(train_model_df["subject"]) & set(holdout_df["subject"]))
)

print("holdout_size:", holdout_size)
print("use_train_subset:", use_train_subset)
print("train_sequence_frac:", train_sequence_frac)

print("\ntrain sequence_type counts:")
print(
    train_model_df
    .drop_duplicates("sequence_id")["sequence_type"]
    .value_counts()
)

print("\nholdout sequence_type counts:")
print(
    holdout_df
    .drop_duplicates("sequence_id")["sequence_type"]
    .value_counts()
)

In [ ]:
# ============================================================
# Common CNN param spaces
# Uses AdvancedMultiDomainSequenceExtractor:
# - multi-domain strings: "raw|velocity|displacement|jerk"
# - motion_filter_mode: None / kalman / extended_kalman
# - use_dead_reckoning: False / True
# ============================================================

if search_mode == "bayesian":
    assert BayesSearchCV is not None, "BayesSearchCV is not available. Set search_mode='grid'."

    base_param_space = {
        f"{pipe_name}__acc_modes": Categorical([
            "raw|velocity|displacement",
            "smoothed|velocity|displacement|jerk",
            "raw|velocity|displacement|jerk",
        ]),
        f"{pipe_name}__linear_acc_mode": Categorical([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([False, True]),
        f"{pipe_name}__sampling_rate": Categorical([10, 20, 25]),
        f"{pipe_name}__clip_value": Categorical([None, 20.0, 50.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear"]),
        f"{pipe_name}__window_size": Integer(5, 31),
        f"{pipe_name}__smooth_alpha": Categorical([None, 0.5, 0.8]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__rotation_modes": Categorical([
            "quaternion|rot6d",
            "rot6d|angular_velocity",
            "quaternion|rot6d|angular_velocity",
            "quaternion|euler|rot6d|angular_velocity",
        ]),
        f"{pipe_name}__tof_mode": Categorical([None, "sensor_stats", "pooled_stats", "pooled_diff"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["nan_interpolate", "far_255", "far_500"]),
        f"{pipe_name}__thm_mode": Categorical([None, "centered", "centered_diff"]),
        f"{pipe_name}__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-4, 2.0, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([False, True]),

        f"{classifier_name}__maxlen": Integer(64, 180),
        f"{classifier_name}__conv_filters": Categorical(["64-128", "128-256", "128-256-256"]),
        f"{classifier_name}__kernel_sizes": Categorical(["3-3", "5-5", "5-5-5"]),
        f"{classifier_name}__pool_sizes": Categorical(["none", "2", "2-2", "2-2-2"]),
        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Real(0.0, 0.5),
        f"{classifier_name}__dense_units": Categorical(["16", "32", "64", "128"]),
        f"{classifier_name}__dropout": Real(0.0, 0.6),
        f"{classifier_name}__learning_rate": Real(1e-6, 2e-3, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([16, 32, 64]),
        f"{classifier_name}__epochs": Categorical([80, 120, 200]),
        f"{classifier_name}__patience": Categorical([10, 20]),

        f"{classifier_name}__use_mixup": Categorical([False, True]),
        f"{classifier_name}__mixup_alpha": Real(0.2, 0.6),
        f"{classifier_name}__mixup_size": Real(0.5, 1.5),
        f"{classifier_name}__use_gaussian_noise": Categorical([False, True]),
        f"{classifier_name}__noise_std": Real(0.001, 0.05, prior="log-uniform"),
        f"{classifier_name}__use_time_mask": Categorical([False, True]),
        f"{classifier_name}__time_mask_ratio": Real(0.05, 0.2),
        f"{classifier_name}__use_time_shift": Categorical([False, True]),
        f"{classifier_name}__max_shift_pct": Real(0.05, 0.3),
        f"{classifier_name}__use_time_stretch": Categorical([False, True]),
        f"{classifier_name}__time_stretch_min_rate": Real(0.7, 0.95),
        f"{classifier_name}__time_stretch_max_rate": Real(1.05, 1.3),
        f"{classifier_name}__use_magnitude_scaling": Categorical([False, True]),
        f"{classifier_name}__use_channel_dropout": Categorical([False, True]),
        f"{classifier_name}__channel_dropout_prob": Real(0.0, 0.25),
        f"{classifier_name}__use_modality_dropout": Categorical([False, True]),
        f"{classifier_name}__drop_tof_prob": Real(0.0, 0.4),
        f"{classifier_name}__drop_thm_prob": Real(0.0, 0.4),
    }

    target_param_space = base_param_space.copy()

    orientation_param_space = base_param_space.copy()
    orientation_param_space.update({
        f"{pipe_name}__acc_modes": Categorical([
            "displacement",
            "velocity|displacement",
            "raw|velocity|displacement",
            "smoothed|velocity|displacement|jerk",
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "rot6d",
            "quaternion|rot6d",
            "quaternion|rot6d|angular_velocity",
        ]),
    })

    action_param_space = base_param_space.copy()
    action_param_space.update({
        f"{pipe_name}__acc_modes": Categorical([
            "raw|jerk",
            "velocity|jerk",
            "raw|velocity|jerk",
            "smoothed|velocity|displacement|jerk",
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "angular_velocity",
            "delta_euler|angular_velocity",
            "quaternion|rot6d|angular_velocity",
        ]),
    })

    position_param_space = base_param_space.copy()
    position_param_space.update({
        f"{pipe_name}__acc_modes": Categorical([
            "raw|velocity|displacement",
            "smoothed|velocity|displacement|jerk",
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "quaternion|rot6d",
            "quaternion|euler|rot6d|angular_velocity",
        ]),
        f"{pipe_name}__tof_mode": Categorical(["sensor_stats", "pooled_stats", "pooled_diff"]),
    })

elif search_mode == "grid":
    target_param_space = {
        f"{pipe_name}__acc_modes": ["smoothed|velocity|displacement|jerk"],
        f"{pipe_name}__linear_acc_mode": ["baseline"],
        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],
        f"{pipe_name}__sampling_rate": [10],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__window_size": [30],
        f"{pipe_name}__smooth_alpha": [0.8],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__rotation_modes": ["quaternion|rot6d|angular_velocity"],
        f"{pipe_name}__tof_mode": ["pooled_stats"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate"],
        f"{pipe_name}__thm_mode": ["centered"],
        f"{pipe_name}__motion_filter_mode": ["kalman"],
        f"{pipe_name}__kalman_process_noise": [1e-3],
        f"{pipe_name}__kalman_measurement_noise": [1e-1],
        f"{pipe_name}__use_dead_reckoning": [True],

        f"{classifier_name}__maxlen": [160],
        f"{classifier_name}__conv_filters": ["128-256-256"],
        f"{classifier_name}__kernel_sizes": ["5-5-5"],
        f"{classifier_name}__pool_sizes": ["2-2-2"],
        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__spatial_dropout": [0.35],
        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.45],
        f"{classifier_name}__learning_rate": [5e-5],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [120],
        f"{classifier_name}__patience": [20],

        f"{classifier_name}__use_mixup": [True],
        f"{classifier_name}__mixup_alpha": [0.4],
        f"{classifier_name}__mixup_size": [1.0],
        f"{classifier_name}__use_gaussian_noise": [False],
        f"{classifier_name}__use_time_mask": [False],
        f"{classifier_name}__use_time_shift": [False],
        f"{classifier_name}__use_time_stretch": [False],
        f"{classifier_name}__use_magnitude_scaling": [False],
        f"{classifier_name}__use_channel_dropout": [False],
        f"{classifier_name}__use_modality_dropout": [False],
    }

    orientation_param_space = target_param_space.copy()
    orientation_param_space.update({
        f"{pipe_name}__acc_modes": ["velocity|displacement", "smoothed|velocity|displacement|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|rot6d", "quaternion|rot6d|angular_velocity"],
    })

    action_param_space = target_param_space.copy()
    action_param_space.update({
        f"{pipe_name}__acc_modes": ["raw|velocity|jerk", "smoothed|velocity|displacement|jerk"],
        f"{pipe_name}__rotation_modes": ["delta_euler|angular_velocity", "quaternion|rot6d|angular_velocity"],
    })

    position_param_space = target_param_space.copy()
    position_param_space.update({
        f"{pipe_name}__acc_modes": ["raw|velocity|displacement", "smoothed|velocity|displacement|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|rot6d", "quaternion|euler|rot6d|angular_velocity"],
        f"{pipe_name}__tof_mode": ["sensor_stats", "pooled_stats"],
    })

else:
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

In [ ]:
# ============================================================
# Optional: load previous best params from summary CSV
# ============================================================

previous_best_params = {
    "target": None,
    "orientation": None,
    "action": None,
    "position": None,
}

if previous_summary_path is not None:
    previous_summary_df = pd.read_csv(previous_summary_path)
    previous_summary_row = previous_summary_df.iloc[-1]

    previous_best_params["target"] = json.loads(previous_summary_row["target_best_params"])
    previous_best_params["orientation"] = json.loads(previous_summary_row["orientation_best_params"])
    previous_best_params["action"] = json.loads(previous_summary_row["action_best_params"])
    previous_best_params["position"] = json.loads(previous_summary_row["position_best_params"])

    print("Loaded previous best params from:", previous_summary_path)
    print("available previous params:", list(previous_best_params.keys()))
else:
    print("No previous summary path supplied. Search modes will run normally unless model_run_mode uses 'model'.")

In [ ]:
# ============================================================
# Target vs Non-Target model
# ============================================================

target_name = "is_target"

pipeline_target = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.AdvancedMultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_target = GroupKFold(n_splits=n_cv_splits)

y_target = train_model_df[["sequence_id", target_name]].copy()
groups_target = train_model_df["subject"].copy()

if model_run_mode["target"] == "model":
    if previous_model_paths["target"] is None:
        raise ValueError("previous_model_paths['target'] must be set when model_run_mode['target'] == 'model'")

    loaded_estimator = joblib.load(previous_model_paths["target"])
    search_target = SimpleNamespace(
        best_estimator_=loaded_estimator,
        best_params_=getattr(loaded_estimator, "get_params", lambda: {})(),
        best_score_=np.nan,
        cv_results_={"params": ["loaded_model"], "mean_test_score": [np.nan]},
    )
    target_results_df = pd.DataFrame(search_target.cv_results_)
    print("loaded fitted target model from:", previous_model_paths["target"])

elif model_run_mode["target"] == "best_params":
    if previous_best_params["target"] is None:
        raise ValueError("previous_best_params['target'] is empty. Set previous_summary_path first.")

    pipeline_target.set_params(**previous_best_params["target"])
    pipeline_target.fit(train_model_df, y_target)

    refit_score = pipeline_target.score(train_model_df, y_target)
    search_target = SimpleNamespace(
        best_estimator_=pipeline_target,
        best_params_=previous_best_params["target"],
        best_score_=refit_score,
        cv_results_={"params": [previous_best_params["target"]], "mean_test_score": [refit_score]},
    )
    target_results_df = pd.DataFrame(search_target.cv_results_)
    print("loaded previous best params and refit target model")
    print("train refit score:", refit_score)

elif model_run_mode["target"] == "search":
    if search_mode == "bayesian":
        search_target = BayesSearchCV(
            estimator=pipeline_target,
            search_spaces=target_param_space,
            n_iter=n_iter_bayes,
            scoring=None,
            cv=cv_target,
            n_jobs=n_jobs,
            refit=True,
            random_state=random_state,
            verbose=2,
            error_score="raise",
        )
    elif search_mode == "grid":
        search_target = GridSearchCV(
            estimator=pipeline_target,
            param_grid=target_param_space,
            scoring=None,
            cv=cv_target,
            n_jobs=n_jobs,
            refit=True,
            verbose=2,
            error_score="raise",
        )
    else:
        raise ValueError(f"Unknown search_mode: {search_mode}")

    search_target.fit(train_model_df, y_target, groups=groups_target)
    target_results_df = pd.DataFrame(search_target.cv_results_)

else:
    raise ValueError("model_run_mode['target'] must be 'search', 'best_params', or 'model'")

target_results_df_path = results_dir / f"cv_results_target_vs_non_target_{timestamp}.csv"

if model_run_mode["target"] == "search" or save_loaded_model_stub_results:
    target_results_df.to_csv(target_results_df_path, index=False)

if save_fitted_models and model_run_mode["target"] != "model":
    fitted_model_path = results_dir / f"fitted_target_vs_non_target_{timestamp}.joblib"
    joblib.dump(search_target.best_estimator_, fitted_model_path)
    print("saved fitted target model:", fitted_model_path)

print("best is_target score:", search_target.best_score_)
print("best target params:")
display(pd.Series(search_target.best_params_))
print("saved cv/stub results:", target_results_df_path)

In [ ]:
# ============================================================
# Orientation model, Target only
# ============================================================

target_name = "orientation"

pipeline_orientation = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.AdvancedMultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_orientation = GroupKFold(n_splits=n_cv_splits)

y_orientation = target_only_train_df[["sequence_id", target_name]].copy()
groups_orientation = target_only_train_df["subject"].copy()

if model_run_mode["orientation"] == "model":
    if previous_model_paths["orientation"] is None:
        raise ValueError("previous_model_paths['orientation'] must be set when model_run_mode['orientation'] == 'model'")

    loaded_estimator = joblib.load(previous_model_paths["orientation"])
    search_orientation = SimpleNamespace(
        best_estimator_=loaded_estimator,
        best_params_=getattr(loaded_estimator, "get_params", lambda: {})(),
        best_score_=np.nan,
        cv_results_={"params": ["loaded_model"], "mean_test_score": [np.nan]},
    )
    orientation_results_df = pd.DataFrame(search_orientation.cv_results_)
    print("loaded fitted orientation model from:", previous_model_paths["orientation"])

elif model_run_mode["orientation"] == "best_params":
    if previous_best_params["orientation"] is None:
        raise ValueError("previous_best_params['orientation'] is empty. Set previous_summary_path first.")

    pipeline_orientation.set_params(**previous_best_params["orientation"])
    pipeline_orientation.fit(target_only_train_df, y_orientation)

    refit_score = pipeline_orientation.score(target_only_train_df, y_orientation)
    search_orientation = SimpleNamespace(
        best_estimator_=pipeline_orientation,
        best_params_=previous_best_params["orientation"],
        best_score_=refit_score,
        cv_results_={"params": [previous_best_params["orientation"]], "mean_test_score": [refit_score]},
    )
    orientation_results_df = pd.DataFrame(search_orientation.cv_results_)
    print("loaded previous best params and refit orientation model")
    print("train refit score:", refit_score)

elif model_run_mode["orientation"] == "search":
    if search_mode == "bayesian":
        search_orientation = BayesSearchCV(
            estimator=pipeline_orientation,
            search_spaces=orientation_param_space,
            n_iter=n_iter_bayes,
            scoring=None,
            cv=cv_orientation,
            n_jobs=n_jobs,
            refit=True,
            random_state=random_state,
            verbose=2,
            error_score="raise",
        )
    elif search_mode == "grid":
        search_orientation = GridSearchCV(
            estimator=pipeline_orientation,
            param_grid=orientation_param_space,
            scoring=None,
            cv=cv_orientation,
            n_jobs=n_jobs,
            refit=True,
            verbose=2,
            error_score="raise",
        )
    else:
        raise ValueError(f"Unknown search_mode: {search_mode}")

    search_orientation.fit(target_only_train_df, y_orientation, groups=groups_orientation)
    orientation_results_df = pd.DataFrame(search_orientation.cv_results_)

else:
    raise ValueError("model_run_mode['orientation'] must be 'search', 'best_params', or 'model'")

orientation_results_df_path = results_dir / f"cv_results_orientation_{timestamp}.csv"

if model_run_mode["orientation"] == "search" or save_loaded_model_stub_results:
    orientation_results_df.to_csv(orientation_results_df_path, index=False)

if save_fitted_models and model_run_mode["orientation"] != "model":
    fitted_model_path = results_dir / f"fitted_orientation_{timestamp}.joblib"
    joblib.dump(search_orientation.best_estimator_, fitted_model_path)
    print("saved fitted orientation model:", fitted_model_path)

print("best orientation score:", search_orientation.best_score_)
print("best orientation params:")
display(pd.Series(search_orientation.best_params_))
print("saved cv/stub results:", orientation_results_df_path)

In [ ]:
# ============================================================
# Gesture action model, Target only
# ============================================================

target_name = "gesture_action"

pipeline_action = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.AdvancedMultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_action = GroupKFold(n_splits=n_cv_splits)

y_action = target_only_train_df[["sequence_id", target_name]].copy()
groups_action = target_only_train_df["subject"].copy()

if model_run_mode["action"] == "model":
    if previous_model_paths["action"] is None:
        raise ValueError("previous_model_paths['action'] must be set when model_run_mode['action'] == 'model'")

    loaded_estimator = joblib.load(previous_model_paths["action"])
    search_action = SimpleNamespace(
        best_estimator_=loaded_estimator,
        best_params_=getattr(loaded_estimator, "get_params", lambda: {})(),
        best_score_=np.nan,
        cv_results_={"params": ["loaded_model"], "mean_test_score": [np.nan]},
    )
    action_results_df = pd.DataFrame(search_action.cv_results_)
    print("loaded fitted action model from:", previous_model_paths["action"])

elif model_run_mode["action"] == "best_params":
    if previous_best_params["action"] is None:
        raise ValueError("previous_best_params['action'] is empty. Set previous_summary_path first.")

    pipeline_action.set_params(**previous_best_params["action"])
    pipeline_action.fit(target_only_train_df, y_action)

    refit_score = pipeline_action.score(target_only_train_df, y_action)
    search_action = SimpleNamespace(
        best_estimator_=pipeline_action,
        best_params_=previous_best_params["action"],
        best_score_=refit_score,
        cv_results_={"params": [previous_best_params["action"]], "mean_test_score": [refit_score]},
    )
    action_results_df = pd.DataFrame(search_action.cv_results_)
    print("loaded previous best params and refit action model")
    print("train refit score:", refit_score)

elif model_run_mode["action"] == "search":
    if search_mode == "bayesian":
        search_action = BayesSearchCV(
            estimator=pipeline_action,
            search_spaces=action_param_space,
            n_iter=n_iter_bayes,
            scoring=None,
            cv=cv_action,
            n_jobs=n_jobs,
            refit=True,
            random_state=random_state,
            verbose=2,
            error_score="raise",
        )
    elif search_mode == "grid":
        search_action = GridSearchCV(
            estimator=pipeline_action,
            param_grid=action_param_space,
            scoring=None,
            cv=cv_action,
            n_jobs=n_jobs,
            refit=True,
            verbose=2,
            error_score="raise",
        )
    else:
        raise ValueError(f"Unknown search_mode: {search_mode}")

    search_action.fit(target_only_train_df, y_action, groups=groups_action)
    action_results_df = pd.DataFrame(search_action.cv_results_)

else:
    raise ValueError("model_run_mode['action'] must be 'search', 'best_params', or 'model'")

action_results_df_path = results_dir / f"cv_results_gesture_action_{timestamp}.csv"

if model_run_mode["action"] == "search" or save_loaded_model_stub_results:
    action_results_df.to_csv(action_results_df_path, index=False)

if save_fitted_models and model_run_mode["action"] != "model":
    fitted_model_path = results_dir / f"fitted_gesture_action_{timestamp}.joblib"
    joblib.dump(search_action.best_estimator_, fitted_model_path)
    print("saved fitted action model:", fitted_model_path)

print("best gesture_action score:", search_action.best_score_)
print("best action params:")
display(pd.Series(search_action.best_params_))
print("saved cv/stub results:", action_results_df_path)

In [ ]:
# ============================================================
# Gesture position model, Target only
# ============================================================

target_name = "gesture_position"

pipeline_position = Pipeline([
    (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
    (pipe_name, utils.AdvancedMultiDomainSequenceExtractor()),
    (classifier_name, utils.KerasAugmentedCNN1DSequenceClassifier(
        target=target_name,
        verbose=0,
        random_state=random_state,
    )),
])

cv_position = GroupKFold(n_splits=n_cv_splits)

y_position = target_only_train_df[["sequence_id", target_name]].copy()
groups_position = target_only_train_df["subject"].copy()

if model_run_mode["position"] == "model":
    if previous_model_paths["position"] is None:
        raise ValueError("previous_model_paths['position'] must be set when model_run_mode['position'] == 'model'")

    loaded_estimator = joblib.load(previous_model_paths["position"])
    search_position = SimpleNamespace(
        best_estimator_=loaded_estimator,
        best_params_=getattr(loaded_estimator, "get_params", lambda: {})(),
        best_score_=np.nan,
        cv_results_={"params": ["loaded_model"], "mean_test_score": [np.nan]},
    )
    position_results_df = pd.DataFrame(search_position.cv_results_)
    print("loaded fitted position model from:", previous_model_paths["position"])

elif model_run_mode["position"] == "best_params":
    if previous_best_params["position"] is None:
        raise ValueError("previous_best_params['position'] is empty. Set previous_summary_path first.")

    pipeline_position.set_params(**previous_best_params["position"])
    pipeline_position.fit(target_only_train_df, y_position)

    refit_score = pipeline_position.score(target_only_train_df, y_position)
    search_position = SimpleNamespace(
        best_estimator_=pipeline_position,
        best_params_=previous_best_params["position"],
        best_score_=refit_score,
        cv_results_={"params": [previous_best_params["position"]], "mean_test_score": [refit_score]},
    )
    position_results_df = pd.DataFrame(search_position.cv_results_)
    print("loaded previous best params and refit position model")
    print("train refit score:", refit_score)

elif model_run_mode["position"] == "search":
    if search_mode == "bayesian":
        search_position = BayesSearchCV(
            estimator=pipeline_position,
            search_spaces=position_param_space,
            n_iter=n_iter_bayes,
            scoring=None,
            cv=cv_position,
            n_jobs=n_jobs,
            refit=True,
            random_state=random_state,
            verbose=2,
            error_score="raise",
        )
    elif search_mode == "grid":
        search_position = GridSearchCV(
            estimator=pipeline_position,
            param_grid=position_param_space,
            scoring=None,
            cv=cv_position,
            n_jobs=n_jobs,
            refit=True,
            verbose=2,
            error_score="raise",
        )
    else:
        raise ValueError(f"Unknown search_mode: {search_mode}")

    search_position.fit(target_only_train_df, y_position, groups=groups_position)
    position_results_df = pd.DataFrame(search_position.cv_results_)

else:
    raise ValueError("model_run_mode['position'] must be 'search', 'best_params', or 'model'")

position_results_df_path = results_dir / f"cv_results_gesture_position_{timestamp}.csv"

if model_run_mode["position"] == "search" or save_loaded_model_stub_results:
    position_results_df.to_csv(position_results_df_path, index=False)

if save_fitted_models and model_run_mode["position"] != "model":
    fitted_model_path = results_dir / f"fitted_gesture_position_{timestamp}.joblib"
    joblib.dump(search_position.best_estimator_, fitted_model_path)
    print("saved fitted position model:", fitted_model_path)

print("best gesture_position score:", search_position.best_score_)
print("best position params:")
display(pd.Series(search_position.best_params_))
print("saved cv/stub results:", position_results_df_path)

In [ ]:
# ============================================================
# Holdout evaluation: combine hierarchy
# ============================================================

holdout_seq = (
    holdout_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "hierarchical_gesture", "gesture_action", "gesture_position", "orientation"]]
    .reset_index(drop=True)
)

target_holdout_seq = (
    target_only_holdout_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "gesture", "gesture_action", "gesture_position", "orientation"]]
    .reset_index(drop=True)
)

is_target_pred = search_target.best_estimator_.predict(holdout_df)
orientation_pred_all = search_orientation.best_estimator_.predict(holdout_df)
action_pred_all = search_action.best_estimator_.predict(holdout_df)
position_pred_all = search_position.best_estimator_.predict(holdout_df)

holdout_seq.loc[:, "pred_is_target"] = is_target_pred.astype(int)
holdout_seq.loc[:, "pred_orientation"] = orientation_pred_all.astype(str)
holdout_seq.loc[:, "pred_gesture_action"] = action_pred_all.astype(str)
holdout_seq.loc[:, "pred_gesture_position"] = position_pred_all.astype(str)

combo_keys = list(zip(
    holdout_seq["pred_gesture_position"].astype(str),
    holdout_seq["pred_gesture_action"].astype(str),
))

mapped_target_pred = pd.Series(combo_keys).map(position_action_lookup)
invalid_combo_count = int(mapped_target_pred.isna().sum())
mapped_target_pred = mapped_target_pred.fillna(most_common_target_gesture)

holdout_seq.loc[:, "prediction"] = np.where(
    holdout_seq["pred_is_target"].eq(1),
    mapped_target_pred.to_numpy(),
    "Non-Target",
)

hier_f1 = f1_score(
    holdout_seq["hierarchical_gesture"],
    holdout_seq["prediction"],
    average="macro",
)

target_detector_f1 = f1_score(
    holdout_seq["sequence_type"].eq("Target").astype(int),
    holdout_seq["pred_is_target"].astype(int),
    average="macro",
)

orientation_pred_target = search_orientation.best_estimator_.predict(target_only_holdout_df)
action_pred_target = search_action.best_estimator_.predict(target_only_holdout_df)
position_pred_target = search_position.best_estimator_.predict(target_only_holdout_df)

target_holdout_seq.loc[:, "pred_orientation"] = orientation_pred_target.astype(str)
target_holdout_seq.loc[:, "pred_gesture_action"] = action_pred_target.astype(str)
target_holdout_seq.loc[:, "pred_gesture_position"] = position_pred_target.astype(str)

target_combo_keys = list(zip(
    target_holdout_seq["pred_gesture_position"].astype(str),
    target_holdout_seq["pred_gesture_action"].astype(str),
))

target_holdout_seq.loc[:, "reconstructed_gesture_pred"] = (
    pd.Series(target_combo_keys).map(position_action_lookup).fillna(most_common_target_gesture).to_numpy()
)

orientation_f1 = f1_score(
    target_holdout_seq["orientation"],
    target_holdout_seq["pred_orientation"],
    average="macro",
)

gesture_action_f1 = f1_score(
    target_holdout_seq["gesture_action"],
    target_holdout_seq["pred_gesture_action"],
    average="macro",
)

gesture_position_f1 = f1_score(
    target_holdout_seq["gesture_position"],
    target_holdout_seq["pred_gesture_position"],
    average="macro",
)

reconstructed_target_gesture_f1 = f1_score(
    target_holdout_seq["gesture"],
    target_holdout_seq["reconstructed_gesture_pred"],
    average="macro",
)

print("Target detector holdout macro F1:", round(target_detector_f1, 4))
print("Orientation holdout macro F1:", round(orientation_f1, 4))
print("Gesture action holdout macro F1:", round(gesture_action_f1, 4))
print("Gesture position holdout macro F1:", round(gesture_position_f1, 4))
print("Reconstructed target gesture holdout macro F1:", round(reconstructed_target_gesture_f1, 4))
print("Full hierarchical holdout macro F1:", round(hier_f1, 4))
print("Invalid target combos fallback count:", invalid_combo_count)

print("\nTarget detector report")
print(classification_report(
    holdout_seq["sequence_type"].eq("Target").astype(int),
    holdout_seq["pred_is_target"].astype(int),
))

print("\nOrientation report")
print(classification_report(
    target_holdout_seq["orientation"],
    target_holdout_seq["pred_orientation"],
))

print("\nGesture action report")
print(classification_report(
    target_holdout_seq["gesture_action"],
    target_holdout_seq["pred_gesture_action"],
))

print("\nGesture position report")
print(classification_report(
    target_holdout_seq["gesture_position"],
    target_holdout_seq["pred_gesture_position"],
))

print("\nReconstructed target gesture report")
print(classification_report(
    target_holdout_seq["gesture"],
    target_holdout_seq["reconstructed_gesture_pred"],
))

print("\nFull hierarchy report")
print(classification_report(
    holdout_seq["hierarchical_gesture"],
    holdout_seq["prediction"],
))

holdout_results_path = results_dir / f"holdout_results_four_stage_hierarchy_cnn_{timestamp}.csv"
target_holdout_results_path = results_dir / f"target_only_holdout_results_four_stage_hierarchy_cnn_{timestamp}.csv"

holdout_seq.to_csv(holdout_results_path, index=False)
target_holdout_seq.to_csv(target_holdout_results_path, index=False)

print("saved:", holdout_results_path)
print("saved:", target_holdout_results_path)

In [ ]:
# ============================================================
# Save compact summary
# ============================================================

summary_df = pd.DataFrame([
    {
        "timestamp": timestamp,
        "search_mode": search_mode,
        "holdout_size": holdout_size,
        "n_cv_splits": n_cv_splits,
        "target_run_mode": model_run_mode["target"],
        "orientation_run_mode": model_run_mode["orientation"],
        "action_run_mode": model_run_mode["action"],
        "position_run_mode": model_run_mode["position"],
        "target_cv_best_score": search_target.best_score_,
        "orientation_cv_best_score": search_orientation.best_score_,
        "gesture_action_cv_best_score": search_action.best_score_,
        "gesture_position_cv_best_score": search_position.best_score_,
        "target_detector_holdout_macro_f1": target_detector_f1,
        "orientation_holdout_macro_f1": orientation_f1,
        "gesture_action_holdout_macro_f1": gesture_action_f1,
        "gesture_position_holdout_macro_f1": gesture_position_f1,
        "reconstructed_target_gesture_holdout_macro_f1": reconstructed_target_gesture_f1,
        "full_hierarchical_holdout_macro_f1": hier_f1,
        "invalid_combo_count": invalid_combo_count,
        "train_sequences": train_model_df["sequence_id"].nunique(),
        "holdout_sequences": holdout_df["sequence_id"].nunique(),
        "target_train_sequences": target_only_train_df["sequence_id"].nunique(),
        "target_holdout_sequences": target_only_holdout_df["sequence_id"].nunique(),
        "train_subjects": train_model_df["subject"].nunique(),
        "holdout_subjects": holdout_df["subject"].nunique(),
        "target_best_params": json.dumps(search_target.best_params_, default=str),
        "orientation_best_params": json.dumps(search_orientation.best_params_, default=str),
        "action_best_params": json.dumps(search_action.best_params_, default=str),
        "position_best_params": json.dumps(search_position.best_params_, default=str),
    }
])

summary_path = results_dir / f"summary_four_stage_hierarchy_cnn_{timestamp}.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print("saved:", summary_path)